# 02 — Gate 1: data spine + cross-dataset dedup audit
Hard go/no-go. Pull three role datasets, harmonise, check whether a transport gap would be real shift or an artefact of shared near-duplicate prompts.

direct+benign -> `deepset/prompt-injections` · over-defense -> `leolee99/NotInject` · indirect -> `microsoft/BIPIA` (best-effort).

Dedup = MinHash + LSH at Jaccard >= 0.8 (Gate AI protocol, arXiv 2606.02959; cite, not novel).

## Session bootstrap

In [1]:
# === SESSION BOOTSTRAP (run first, every session) ===
# A fresh Colab runtime forgets --global git config, so we re-set it here.
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')          # so `import metrics` finds src/metrics.py
!git pull
print('Session ready - identity set, src/ on path, repo pulled.')

Mounted at /content/drive
/content/drive/MyDrive/PICALIB_Research/picalib-research
Already up to date.
Session ready - identity set, src/ on path, repo pulled.


## Dependencies

In [ ]:
!pip install -q datasets datasketch pandas
print('deps installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.9/99.9 kB 1.6 MB/s eta 0:00:00


## Load + harmonise (never hard-fails on one source)

In [ ]:
import importlib, data_loaders, dedup
importlib.reload(data_loaders); importlib.reload(dedup)
from data_loaders import load_all
import json, pandas as pd

df, report = load_all(include_bipia=True)
print('\nLoad report:'); print(json.dumps(report, indent=2))
print('\nsource / unit_type / label:')
print(df.groupby(['source','unit_type','label']).size())

## Detection-unit manifest (standalone vs in-document)

In [ ]:
import os
os.makedirs('data', exist_ok=True); os.makedirs('reports', exist_ok=True)
df.to_csv('data/harmonised.csv', index=False)   # gitignored
manifest = df.groupby(['source','unit_type','label']).size().reset_index(name='n')
manifest.to_csv('reports/gate1_manifest.csv', index=False)
print(manifest.to_string(index=False))

## Cross-dataset near-duplicate audit (the go/no-go)

In [ ]:
from dedup import dedup_audit, print_report
audit = dedup_audit(df, threshold=0.8, k=3)
print_report(audit)

## Save audit + verdict

In [ ]:
audit['overlap_frac'].to_csv('reports/gate1_overlap_frac.csv')
v, reason = audit['verdict']
open('reports/gate1_verdict.txt','w').write(f'{v} - {reason}\n')
print('GATE 1:', v)
if v != 'GO':
    print('-> inspect example pairs above; swap a backup source before scaling up.')

---
## Commit + push

In [ ]:
!git add -A
!git commit -m "Gate 1: data spine + MinHash-LSH cross-dataset dedup audit"
!git push
print('Pushed. Confirm GitHub + Drive in sync.')